# NB_PERF_01 · Diagnosticar el modelo semántico

**Paso 1 del framework de performance. Solo lectura**: no modifica el modelo.

Junta en una corrida (por ejemplo `antes` o `despues_pr12`) todo lo que Claude necesita para proponer mejoras:

| Archivo en `Files/doc_modelos/<modelo>/perf/<corrida>/` | Qué tiene | De dónde sale |
|---|---|---|
| `vertipaq_modelo.csv`, `vertipaq_tablas.csv`, `vertipaq_columnas.csv`, `vertipaq_relaciones.csv`, `vertipaq_particiones.csv` | Memoria, cardinalidad, encoding y tipo de cada objeto | `sempy_labs.vertipaq_analyzer` |
| `bpa.csv` | Reglas del Best Practice Analyzer que no se cumplen | `sempy_labs.run_model_bpa` |
| `fallback.csv` | Solo Direct Lake: motivo por el que una tabla caería a DirectQuery | `sempy_labs.directlake.check_fallback_reason` |
| `diagnostico.json` | Resumen compacto (top de columnas, relaciones, BPA de performance) para pasarle a Claude | Este notebook |
| `corrida.json` | Modelo, workspace, commit medido, fecha y versión de las librerías | Este notebook |

Reutiliza el caso **Documentar_Modelo_Semantico**: si existe `Files/doc_modelos/<modelo>/modelo.json` (lo genera `NB_DOC_01`), suma al diagnóstico las expresiones DAX de las medidas y las descripciones. No vuelve a extraerlas.

Requisitos: lakehouse por defecto adjunto, endpoint XMLA con lectura en la capacidad y permiso **Build/Write** sobre el modelo (igual que `NB_DOC_01`).

In [ ]:
# Parámetros (podés marcar esta celda como "parameters" para llamarla desde un pipeline)
MODELO = "<nombre-del-modelo-semantico>"
WORKSPACE = "<workspace-de-prueba>"     # el workspace donde está desplegada la versión a medir
CORRIDA = "antes"                       # nombre de la corrida: antes, despues_pr12, ...
COMMIT = ""                             # commit del repo PBIP desplegado en ese workspace (para trazabilidad)
CARPETA_BASE = "/lakehouse/default/Files/doc_modelos"   # la misma que usa NB_DOC_01
TOP_COLUMNAS = 30                       # columnas más pesadas que van a diagnostico.json
SOBREESCRIBIR = False                   # True para repetir una corrida con el mismo nombre
GUARDAR_DELTA = True                    # además de los CSV, agrega filas a tablas Delta perf_*

In [ ]:
# Versión fija: los nombres de columnas que devuelve vertipaq_analyzer y run_model_bpa
# se verificaron con 0.17.1. Si la cambiás, corré pruebas/prueba_local.py y revisá la salida.
%pip install -q semantic-link-labs==0.17.1

In [ ]:
# --- funciones (sin dependencias de Fabric: las prueba pruebas/prueba_local.py)
import json
import re

import pandas as pd


def carpeta_modelo(base: str, modelo: str) -> str:
    """Misma convención que NB_DOC_01: Files/doc_modelos/<modelo>/."""
    return base.rstrip("/") + "/" + re.sub(r"[^\w\-]+", "_", modelo)


def normalizar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """'Total Size' -> 'total_size', '% DB' -> 'pct_db'. Delta no acepta espacios ni %."""
    def norm(c):
        c = str(c).replace("%", "pct ")
        return re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower()
    out = df.copy()
    out.columns = [norm(c) for c in out.columns]
    return out


# run_model_bpa(return_dataframe=True) devuelve la severidad como ícono
SEVERIDAD = {"\u26a0\ufe0f": "Warning", "\u274c": "Error", "\u2139\ufe0f": "Info"}


def normalizar_bpa(bpa: pd.DataFrame) -> pd.DataFrame:
    out = bpa.copy()
    out["Severity"] = out["Severity"].map(lambda s: SEVERIDAD.get(s, s))
    return out


def _num(v):
    try:
        return None if pd.isna(v) else float(v)
    except (TypeError, ValueError):
        return None


def armar_diagnostico(vpa: dict, bpa: pd.DataFrame, fallback: pd.DataFrame | None,
                      modelo_doc: dict | None, top_n: int = 30) -> dict:
    """Resumen compacto para Claude: lo que más pesa, lo que rompe reglas y por qué cae Direct Lake.

    vpa: el dict que devuelve vertipaq_analyzer (claves 'Model Summary', 'Tables', 'Columns',
    'Relationships', 'Partitions'). bpa: el DataFrame de run_model_bpa(return_dataframe=True).
    """
    resumen = vpa["Model Summary"].iloc[0].to_dict() if len(vpa["Model Summary"]) else {}
    total = _num(resumen.get("Total Size")) or 0

    cols = vpa["Columns"].sort_values("Total Size", ascending=False).head(top_n)
    columnas = [{
        "tabla": r["Table Name"], "columna": r["Column Name"], "tipo": r.get("Type"),
        "tipo_dato": r.get("Data Type"), "encoding": r.get("Encoding"),
        "cardinalidad": _num(r.get("Cardinality")), "bytes": _num(r.get("Total Size")),
        "pct_modelo": round(100 * _num(r.get("Total Size")) / total, 2) if total else None,
        "diccionario_bytes": _num(r.get("Dictionary Size")),
    } for _, r in cols.iterrows()]

    tablas = [{
        "tabla": r["Table Name"], "tipo": r.get("Type"), "filas": _num(r.get("Row Count")),
        "bytes": _num(r.get("Total Size")), "pct_modelo": _num(r.get("% DB")),
    } for _, r in vpa["Tables"].iterrows()]

    rel = vpa["Relationships"]
    relaciones = [{
        "desde": r["From Object"], "hacia": r["To Object"], "multiplicidad": r.get("Multiplicity"),
        "bytes": _num(r.get("Used Size")), "max_card_desde": _num(r.get("Max From Cardinality")),
        "max_card_hacia": _num(r.get("Max To Cardinality")),
        "filas_sin_match": _num(r.get("Missing Rows")) if "Missing Rows" in rel.columns else None,
    } for _, r in rel.iterrows()]

    modos = sorted({str(m) for m in vpa["Partitions"].get("Mode", pd.Series(dtype=str)).dropna()})

    bpa_perf = bpa[bpa["Category"].astype(str).str.contains("Performance", case=False, na=False)]
    reglas = (bpa.groupby(["Category", "Rule Name", "Severity"]).size()
                 .reset_index(name="objetos").sort_values("objetos", ascending=False))

    diag = {
        "modelo": resumen.get("Dataset Name"),
        "bytes_total": total,
        "tablas_n": _num(resumen.get("Table Count")),
        "columnas_n": _num(resumen.get("Column Count")),
        "compatibilidad": resumen.get("Compatibility Level"),
        "modos_particion": modos,
        "top_columnas": columnas,
        "tablas": tablas,
        "relaciones": relaciones,
        "bpa_resumen": reglas.to_dict("records"),
        "bpa_performance": bpa_perf[["Rule Name", "Object Type", "Object Name", "Severity"]]
                               .to_dict("records"),
        "direct_lake_fallback": [] if fallback is None else fallback.to_dict("records"),
    }
    if modelo_doc:
        # Del caso Documentar_Modelo_Semantico: expresiones y descripciones de las medidas
        diag["medidas"] = [{"tabla": m["tabla"], "medida": m["nombre"], "expresion": m["expresion"],
                            "descripcion": m.get("descripcion_actual", "")}
                           for m in modelo_doc.get("medidas", [])]
        diag["modelo_json_generado_utc"] = modelo_doc.get("generado_utc")
    return diag

## 1. Preparar la carpeta de la corrida

In [ ]:
import os
from datetime import datetime, timezone
from importlib.metadata import version

import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs import directlake

base_modelo = carpeta_modelo(CARPETA_BASE, MODELO)
salida = f"{base_modelo}/perf/{CORRIDA}"
if os.path.exists(f"{salida}/diagnostico.json") and not SOBREESCRIBIR:
    raise FileExistsError(f"La corrida '{CORRIDA}' ya tiene diagnóstico. Usá otro nombre o SOBREESCRIBIR = True.")
os.makedirs(salida, exist_ok=True)

modelo_doc = None
ruta_doc = f"{base_modelo}/modelo.json"
if os.path.exists(ruta_doc):
    with open(ruta_doc, encoding="utf-8") as f:
        modelo_doc = json.load(f)
    print("Uso modelo.json de NB_DOC_01, generado", modelo_doc.get("generado_utc"))
else:
    print("No encontré modelo.json. Recomendado: correr antes NB_DOC_01 del caso Documentar_Modelo_Semantico "
          "para que Claude vea las expresiones DAX de las medidas.")

## 2. Vertipaq Analyzer: dónde está la memoria

`visualize=False` devuelve un dict de DataFrames. `read_stats_from_data=True` calcula además filas sin match en las relaciones (en Direct Lake no). Puede tardar en modelos grandes: si molesta, ponelo en `False`.

In [ ]:
vpa = labs.vertipaq_analyzer(dataset=MODELO, workspace=WORKSPACE, read_stats_from_data=True, visualize=False)

archivos_vpa = {"Model Summary": "vertipaq_modelo", "Tables": "vertipaq_tablas", "Columns": "vertipaq_columnas",
                "Relationships": "vertipaq_relaciones", "Partitions": "vertipaq_particiones"}
for clave, nombre in archivos_vpa.items():
    vpa[clave].to_csv(f"{salida}/{nombre}.csv", index=False, encoding="utf-8")

display(vpa["Model Summary"])
display(vpa["Columns"].sort_values("Total Size", ascending=False).head(15)
        [["Table Name", "Column Name", "Data Type", "Cardinality", "Total Size", "% DB", "Encoding"]])

## 3. Best Practice Analyzer

Usa las reglas por defecto de semantic-link-labs (basadas en las de Microsoft y Tabular Editor). Para reglas propias, armá un DataFrame con `labs.model_bpa_rules()` como base y pasalo en `rules=`.

In [ ]:
bpa = normalizar_bpa(labs.run_model_bpa(dataset=MODELO, workspace=WORKSPACE, return_dataframe=True))
bpa.to_csv(f"{salida}/bpa.csv", index=False, encoding="utf-8")
display(bpa.groupby(["Category", "Severity"]).size().rename("objetos").reset_index())

## 4. Direct Lake: motivos de fallback a DirectQuery

Solo si alguna partición está en modo Direct Lake. Para los límites por SKU, mirá `directlake.get_direct_lake_guardrails()`.

In [ ]:
fallback = None
modos = set(vpa["Partitions"]["Mode"].astype(str)) if "Mode" in vpa["Partitions"].columns else set()
if any("directlake" in m.lower() for m in modos):
    fallback = directlake.check_fallback_reason(dataset=MODELO, workspace=WORKSPACE)
    fallback.to_csv(f"{salida}/fallback.csv", index=False, encoding="utf-8")
    display(fallback)
else:
    print("Sin particiones Direct Lake: no aplica. Modos:", modos)

## 5. Guardar `diagnostico.json`, `corrida.json` y las tablas Delta

In [ ]:
diag = armar_diagnostico(vpa, bpa, fallback, modelo_doc, top_n=TOP_COLUMNAS)
with open(f"{salida}/diagnostico.json", "w", encoding="utf-8") as f:
    json.dump(diag, f, ensure_ascii=False, indent=2, default=str)

corrida = {
    "modelo": MODELO, "workspace": WORKSPACE, "corrida": CORRIDA, "commit": COMMIT,
    "diagnostico_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "semantic_link_labs": version("semantic-link-labs"), "semantic_link_sempy": version("semantic-link-sempy"),
}
ruta_corrida = f"{salida}/corrida.json"
if os.path.exists(ruta_corrida):          # NB_PERF_02 también escribe acá: se combinan
    with open(ruta_corrida, encoding="utf-8") as f:
        corrida = {**json.load(f), **corrida}
with open(ruta_corrida, "w", encoding="utf-8") as f:
    json.dump(corrida, f, ensure_ascii=False, indent=2)

if GUARDAR_DELTA:
    def agregar(df, tabla):
        df = normalizar_columnas(df).astype(str)
        df["modelo"], df["corrida"], df["commit"] = MODELO, CORRIDA, COMMIT
        df["fecha_utc"] = corrida["diagnostico_utc"]
        spark.createDataFrame(df).write.mode("append").option("mergeSchema", "true").saveAsTable(tabla)
    agregar(vpa["Columns"], "perf_vertipaq_columnas")
    agregar(vpa["Tables"], "perf_vertipaq_tablas")
    agregar(bpa, "perf_bpa")

mb = diag["bytes_total"] / 1024 ** 2 if diag["bytes_total"] else 0
print(f"Listo en {salida}")
print(f"Tamaño en memoria: {mb:,.1f} MB · {len(bpa)} hallazgos BPA "
      f"({len(diag['bpa_performance'])} de performance) · modos {diag['modos_particion']}")